In [1]:
!pip install mne

In [2]:
# Step 1: Unmount if already mounted
!fusermount -u /content/drive

# Step 2: Delete the existing folder
!rm -rf /content/drive

# Step 3: Mount again
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
import warnings
warnings.filterwarnings('ignore', message='Channel names are not unique')

In [6]:
"""
SWCNet v4 — Ensemble of SWCNets on Proven v2 Pipeline
======================================================
Key insight from experiments:
  - v2 (simple SWCNet)       = 72.80% avg ✅ works
  - v3 (deeper architecture) = worse    ❌ overfits on 288 trials
  - enhanced (fine-tuning)   = worse    ❌ too complex

Root cause of gap to 80%+:
  Only ~259 trials per fold → high variance between folds
  Single model picks up noise → unstable predictions

Solution: ENSEMBLE
  Train N_ENSEMBLE=5 independent SWCNets per fold (different seeds)
  Average their softmax probabilities before voting
  5 models × variance reduction = ~√5 lower error ≈ 5-8% accuracy gain

Everything else is IDENTICAL to v2:
  - Same SWCNet architecture (proven not to overfit)
  - Same bandpass 4-40 Hz
  - Same sliding window augmentation (train only)
  - Same center window for val
  - Same 10-fold CV
  - Same window voting on eval session
  - Same normalization

Expected: ~80-85% avg Setup II
Runtime : ~3-4x longer than v2 due to ensemble (5 models per fold)
          ~3-4 hours on T4 for all 9 subjects
"""

# ─────────────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import os
import numpy as np
import scipy.io as sio
import scipy.signal as sig_proc
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore', message='Channel names are not unique')

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (confusion_matrix, classification_report,
                              f1_score, accuracy_score)
from sklearn.model_selection import KFold

import mne
mne.set_log_level('WARNING')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
DATA_DIR     = "/content/drive/MyDrive/BCICIV_2a_gdf"       # ← SET YOUR PATH
RESULTS_DIR  = "/content/drive/MyDrive/swcnet_v4_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

N_SUBJECTS   = 9
N_CH         = 22
N_CLS        = 4
FS           = 250

# Trial extraction
PRE_S        = 0.25
POST_S       = 4.25
TRIAL_SAMP   = int((PRE_S + POST_S) * FS)   # 1125

# Sliding window (identical to v2)
WIN_S        = 3.0
STEP_S       = 0.25
N_WIN        = 7
WIN_SAMP     = int(WIN_S  * FS)    # 750
STEP_SAMP    = int(STEP_S * FS)   # 62
CENTER_WIN   = 3

# Bandpass (identical to v2)
FILT_LO      = 4.0
FILT_HI      = 40.0
FILT_ORD     = 5

# Training (identical to v2)
BATCH        = 32
LR           = 0.001
MAX_EP       = 200
PATIENCE     = 30
DROP         = 0.5
SEED         = 42
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Ensemble — the only new parameter
# 5 models per fold: good balance of accuracy gain vs training time
N_ENSEMBLE   = 5

CLASS_NAMES  = ["Left Hand", "Right Hand", "Feet", "Tongue"]

torch.manual_seed(SEED)
np.random.seed(SEED)

# Comparison baselines
PAPER_ACC = {1:78.16, 2:50.44, 3:87.50,
             4:60.94, 5:66.81, 6:56.25,
             7:81.94, 8:82.29, 9:75.35}
V2_ACC    = {1:83.68, 2:49.65, 3:87.50,
             4:68.75, 5:62.85, 6:54.17,
             7:89.24, 8:80.90, 9:78.47}

print(f"Device     : {DEVICE}")
print(f"DataDir    : {DATA_DIR}")
print(f"N_ENSEMBLE : {N_ENSEMBLE} models per fold")


# ─────────────────────────────────────────────────────────────────────────────
# 1. BANDPASS FILTER  (identical to v2)
# ─────────────────────────────────────────────────────────────────────────────

def bandpass(X):
    """Zero-phase Butterworth 4-40 Hz. X: (N, Ch, T)"""
    nyq  = FS / 2.0
    b, a = sig_proc.butter(FILT_ORD,
                           [FILT_LO/nyq, FILT_HI/nyq],
                           btype='band')
    out  = np.empty_like(X)
    for i in range(X.shape[0]):
        for c in range(X.shape[1]):
            out[i, c] = sig_proc.filtfilt(b, a, X[i, c])
    return out


# ─────────────────────────────────────────────────────────────────────────────
# 2. DATA LOADING  (identical to v2)
# ─────────────────────────────────────────────────────────────────────────────

def load_labels(mat_path):
    mat = sio.loadmat(mat_path)
    return mat["classlabel"].flatten().astype(int) - 1


def load_session(gdf_path, mat_path, sid, tag):
    raw  = mne.io.read_raw_gdf(gdf_path, preload=True, verbose=False)
    keep = [ch for ch in raw.ch_names
            if 'EOG' not in ch and 'Status' not in ch
            and 'stim' not in ch.lower()][:N_CH]
    raw.pick_channels(keep)

    events, event_id = mne.events_from_annotations(raw, verbose=False)

    MI       = {'769': 0, '770': 1, '771': 2, '772': 3}
    code_map = {}
    for k, v in event_id.items():
        k2 = str(k).strip().replace('.0', '')
        if k2 in MI:
            code_map[v] = MI[k2]
    if not code_map:
        fb = {7: 0, 8: 1, 9: 2, 10: 3}
        for v in np.unique(events[:, 2]):
            if v in fb:
                code_map[v] = fb[v]
    if not code_map:
        raise ValueError(f"No MI events. Found: {event_id}")

    data = raw.get_data()
    del raw
    pre  = int(PRE_S  * FS)
    post = int(POST_S * FS) + 1

    trials, gdf_lbl = [], []
    for onset, _, code in events:
        if code not in code_map:
            continue
        s, e = onset - pre, onset + post
        if s < 0 or e > data.shape[1]:
            continue
        trials.append(data[:, s:e])
        gdf_lbl.append(code_map[code])

    X     = np.array(trials,  dtype=np.float32)
    y_gdf = np.array(gdf_lbl, dtype=np.int64)
    del data

    y_mat = load_labels(mat_path)
    y     = y_mat if len(y_mat) == len(y_gdf) else y_gdf

    if X.shape[2] != TRIAL_SAMP:
        diff = TRIAL_SAMP - X.shape[2]
        if   diff ==  1: X = np.pad(X, ((0,0),(0,0),(0,1)), mode='edge')
        elif diff == -1: X = X[:, :, :TRIAL_SAMP]

    print(f"  [{tag}] X={X.shape}  y={y.shape}  classes={np.unique(y)}")
    return X, y


def load_subject(sid, data_dir):
    pfx  = os.path.join(data_dir, f"A0{sid}")
    Xtr, ytr = load_session(pfx+"T.gdf", pfx+"T.mat", sid, "train")
    Xev, yev = load_session(pfx+"E.gdf", pfx+"E.mat", sid, "eval")
    return Xtr, ytr, Xev, yev


# ─────────────────────────────────────────────────────────────────────────────
# 3. WINDOWING  (identical to v2)
# ─────────────────────────────────────────────────────────────────────────────

def win_augment(X, y):
    """(N,22,1125) → (N*7,22,750)"""
    Xo, yo = [], []
    for i in range(len(X)):
        for w in range(N_WIN):
            s = w * STEP_SAMP
            e = s + WIN_SAMP
            if e > X.shape[2]: break
            Xo.append(X[i, :, s:e])
            yo.append(y[i])
    return (np.array(Xo, dtype=np.float32),
            np.array(yo, dtype=np.int64))


def win_center(X, y):
    """(N,22,1125) → (N,22,750) center window"""
    s = CENTER_WIN * STEP_SAMP
    e = s + WIN_SAMP
    return X[:, :, s:e].astype(np.float32), y.copy()


# ─────────────────────────────────────────────────────────────────────────────
# 4. NORMALIZATION  (identical to v2)
# ─────────────────────────────────────────────────────────────────────────────

def fit_norm(X):
    mean = X.mean(axis=(0, 2), keepdims=True)
    std  = X.std( axis=(0, 2), keepdims=True) + 1e-8
    return mean, std


def apply_norm(X, mean, std):
    return (X - mean) / std


# ─────────────────────────────────────────────────────────────────────────────
# 5. MODEL: SWCNet  (identical to v2)
# ─────────────────────────────────────────────────────────────────────────────

class SWCNet(nn.Module):
    """
    Exact SWCNet from the paper (same as v2).
    Proven not to overfit on 288-trial datasets.
    """
    def __init__(self, n_ch=N_CH, n_cls=N_CLS, n_t=WIN_SAMP, drop=DROP):
        super().__init__()
        self.sp_conv = nn.Conv2d(1,  16, kernel_size=(n_ch, 1), bias=False)
        self.bn1     = nn.BatchNorm2d(16)
        self.tm_conv = nn.Conv2d(16, 32, kernel_size=(1, 64),
                                 padding=(0, 32), bias=False)
        self.bn2     = nn.BatchNorm2d(32)
        self.act1    = nn.ReLU()
        self.pool1   = nn.AvgPool2d((1, 2))
        self.pw_conv = nn.Conv2d(32, 16, kernel_size=(1, 1), bias=False)
        self.bn3     = nn.BatchNorm2d(16)
        self.act2    = nn.ReLU()
        self.pool2   = nn.AvgPool2d((1, 4))
        self.pool3   = nn.AvgPool2d((1, 8))
        self.dropout = nn.Dropout(p=drop)
        with torch.no_grad():
            flat = self._fwd(torch.zeros(1, 1, n_ch, n_t)).numel()
        self.fc = nn.Linear(flat, n_cls)

    def _fwd(self, x):
        x = self.bn1(self.sp_conv(x))
        x = self.act1(self.bn2(self.tm_conv(x)))
        x = self.pool1(x)
        x = self.act2(self.bn3(self.pw_conv(x)))
        x = self.pool2(x)
        x = self.pool3(x)
        x = self.dropout(x)
        return x

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self._fwd(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


# ─────────────────────────────────────────────────────────────────────────────
# 6. TRAINING UTILITIES  (identical to v2)
# ─────────────────────────────────────────────────────────────────────────────

class EarlyStopping:
    def __init__(self, patience=PATIENCE):
        self.patience = patience
        self.counter  = 0
        self.best     = np.inf
        self.stop     = False
        self.state    = None

    def step(self, val_loss, model):
        if val_loss < self.best - 1e-5:
            self.best    = val_loss
            self.state   = {k: v.clone()
                            for k, v in model.state_dict().items()}
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

    def restore(self, model):
        if self.state:
            model.load_state_dict(self.state)


def make_loader(X, y, shuffle=True):
    ds = TensorDataset(torch.FloatTensor(X), torch.LongTensor(y))
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle, num_workers=0)


def train_ep(model, dl, opt, crit):
    model.train()
    ls, cor, tot = 0., 0, 0
    for xb, yb in dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        out = model(xb)
        l   = crit(out, yb)
        l.backward()
        opt.step()
        ls  += l.item() * len(xb)
        cor += (out.argmax(1) == yb).sum().item()
        tot += len(xb)
    return ls/tot, cor/tot


@torch.no_grad()
def eval_ep(model, dl, crit):
    model.eval()
    ls, cor, tot = 0., 0, 0
    preds, trues = [], []
    for xb, yb in dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        out    = model(xb)
        l      = crit(out, yb)
        ls    += l.item() * len(xb)
        p      = out.argmax(1)
        cor   += (p == yb).sum().item()
        tot   += len(xb)
        preds.extend(p.cpu().numpy())
        trues.extend(yb.cpu().numpy())
    f1 = f1_score(trues, preds, average='macro', zero_division=0)
    return ls/tot, cor/tot, f1, np.array(preds), np.array(trues)


def train_one_model(Xtr, ytr, Xva, yva, seed, tag=""):
    """Train one SWCNet with a given seed. Returns trained model."""
    # Each ensemble member uses a different random seed
    torch.manual_seed(seed)
    np.random.seed(seed)

    tr_dl = make_loader(Xtr, ytr, shuffle=True)
    va_dl = make_loader(Xva, yva, shuffle=False)

    model = SWCNet().to(DEVICE)
    crit  = nn.CrossEntropyLoss()
    opt   = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
                opt, mode='min', factor=0.5, patience=10)
    es    = EarlyStopping(PATIENCE)

    for ep in range(1, MAX_EP + 1):
        tr_l, tr_a = train_ep(model, tr_dl, opt, crit)
        va_l, va_a, _, _, _ = eval_ep(model, va_dl, crit)
        sched.step(va_l)
        es.step(va_l, model)

        if ep % 20 == 0 or ep == 1:
            print(f"    {tag} m{seed} Ep{ep:3d} | "
                  f"Tr {tr_l:.4f}/{tr_a*100:.1f}% "
                  f"Va {va_l:.4f}/{va_a*100:.1f}%")
        if es.stop:
            print(f"    {tag} m{seed} early stop @ ep{ep}")
            break

    es.restore(model)
    return model


# ─────────────────────────────────────────────────────────────────────────────
# 7. ENSEMBLE VOTING INFERENCE  (new in v4)
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def ensemble_predict(models, X_trials, mean, std):
    """
    Ensemble + window voting combined.

    For each trial:
      For each of 7 windows:
        For each of N_ENSEMBLE models:
          Get softmax probabilities → accumulate
      Average all (7 × N_ENSEMBLE) probability vectors
      Take argmax → final prediction

    X_trials : (N, 22, 1125) filtered trials
    models   : list of N_ENSEMBLE trained SWCNet models
    Returns  : (N,) predicted classes
    """
    N         = len(X_trials)
    all_probs = np.zeros((N, N_CLS), dtype=np.float32)
    n_votes   = 0

    for w in range(N_WIN):
        s  = w * STEP_SAMP
        e  = s + WIN_SAMP
        Xw = apply_norm(
            X_trials[:, :, s:e].astype(np.float32), mean, std
        )
        xb = torch.FloatTensor(Xw).to(DEVICE)

        for model in models:
            model.eval()
            probs      = torch.softmax(model(xb), dim=1).cpu().numpy()
            all_probs += probs
            n_votes   += 1

    return (all_probs / n_votes).argmax(axis=1)


# ─────────────────────────────────────────────────────────────────────────────
# 8. SETUP II WITH ENSEMBLE  (v4 main experiment)
# ─────────────────────────────────────────────────────────────────────────────

def setup_II_ensemble(sid, data_dir):
    """
    10-fold CV Setup II but with N_ENSEMBLE models per fold.

    Per fold:
      - Train N_ENSEMBLE SWCNets with seeds SEED+0, SEED+1, ..., SEED+4
      - Select best fold by average val accuracy across ensemble
      - At test time: average softmax over 5 models × 7 windows = 35 votes
    """
    print(f"\n{'='*65}")
    print(f"  Subject {sid}  |  Ensemble SWCNet  "
          f"({N_ENSEMBLE} models/fold)  Setup II")
    print(f"{'='*65}")

    Xtr_raw, ytr, Xev_raw, yev = load_subject(sid, data_dir)

    print("  Bandpass filtering (4-40 Hz)...")
    Xtr_f = bandpass(Xtr_raw)
    Xev_f = bandpass(Xev_raw)
    del Xtr_raw, Xev_raw

    kf = KFold(n_splits=10, shuffle=True, random_state=SEED + sid)
    fold_acc, fold_f1 = [], []
    best_va       = -1
    best_mn       = best_sd = None
    best_ensemble = None   # list of state_dicts for best fold

    for fold, (tri, vai) in enumerate(kf.split(Xtr_f)):
        print(f"\n  ── Fold {fold+1}/10 ──────────────────────────────────")
        Xf_tr, yf_tr = Xtr_f[tri], ytr[tri]
        Xf_va, yf_va = Xtr_f[vai], ytr[vai]

        # Windowing
        Xf_tr_w, yf_tr_w = win_augment(Xf_tr, yf_tr)
        Xf_va_w, yf_va_w = win_center( Xf_va, yf_va)

        # Normalize (fit on this fold's train windows)
        mn, sd   = fit_norm(Xf_tr_w)
        Xf_tr_n  = apply_norm(Xf_tr_w, mn, sd)
        Xf_va_n  = apply_norm(Xf_va_w, mn, sd)

        # ── Train N_ENSEMBLE models with different seeds ──────────────────
        fold_models = []
        fold_va_accs = []
        crit = nn.CrossEntropyLoss()

        for m_idx in range(N_ENSEMBLE):
            seed_m = SEED * 10 + fold * N_ENSEMBLE + m_idx
            print(f"  Training model {m_idx+1}/{N_ENSEMBLE} "
                  f"(seed={seed_m})...")
            m = train_one_model(
                Xf_tr_n, yf_tr_w,
                Xf_va_n, yf_va_w,
                seed=seed_m,
                tag=f"[S{sid} f{fold+1}]"
            )
            _, va_acc, va_f1, _, _ = eval_ep(
                m, make_loader(Xf_va_n, yf_va_w, False), crit
            )
            fold_models.append(m)
            fold_va_accs.append(va_acc)
            print(f"  Model {m_idx+1} | Va Acc={va_acc*100:.2f}%  "
                  f"F1={va_f1:.4f}")

        # Ensemble val accuracy (average softmax over all models)
        all_probs = np.zeros((len(Xf_va_w), N_CLS), dtype=np.float32)
        xb_va     = torch.FloatTensor(Xf_va_n).to(DEVICE)
        for m in fold_models:
            m.eval()
            with torch.no_grad():
                all_probs += torch.softmax(
                    m(xb_va), dim=1
                ).cpu().numpy()
        ens_preds  = (all_probs / N_ENSEMBLE).argmax(axis=1)
        ens_va_acc = accuracy_score(yf_va_w, ens_preds)
        ens_va_f1  = f1_score(yf_va_w, ens_preds,
                               average='macro', zero_division=0)

        fold_acc.append(ens_va_acc)
        fold_f1.append(ens_va_f1)
        print(f"\n  Fold {fold+1:2d} | Ensemble Va Acc={ens_va_acc*100:.2f}%  "
              f"F1={ens_va_f1:.4f}  "
              f"(individual avg={np.mean(fold_va_accs)*100:.2f}%)")

        # Keep best fold's ensemble
        if ens_va_acc > best_va:
            best_va  = ens_va_acc
            best_mn, best_sd = mn.copy(), sd.copy()
            best_ensemble = [
                {k: v.clone() for k, v in m.state_dict().items()}
                for m in fold_models
            ]

        # Free fold models
        del fold_models

    print(f"\n  CV (ensemble): {np.mean(fold_acc)*100:.2f}% ± "
          f"{np.std(fold_acc)*100:.2f}%")

    # ── Final test: ensemble × window voting on eval session ─────────────
    # Reconstruct best fold's ensemble
    test_models = []
    for state in best_ensemble:
        m = SWCNet().to(DEVICE)
        m.load_state_dict(state)
        test_models.append(m)

    yp = ensemble_predict(test_models, Xev_f, best_mn, best_sd)
    yt = yev

    te_acc = accuracy_score(yt, yp)
    te_f1  = f1_score(yt, yp, average='macro', zero_division=0)

    print(f"\n  ✓ S{sid} | Acc={te_acc*100:.2f}%  F1={te_f1:.4f}")
    print(f"    v2 baseline : {V2_ACC[sid]:.2f}%")
    print(f"    Paper target: {PAPER_ACC[sid]:.2f}%")
    delta_v2  = te_acc*100 - V2_ACC[sid]
    delta_pap = te_acc*100 - PAPER_ACC[sid]
    sym_v2  = "✅" if delta_v2  > 0 else "⚠️ "
    sym_pap = "✅" if delta_pap > 0 else "⚠️ "
    print(f"    Δ vs v2     : {sym_v2}{delta_v2:+.2f}%")
    print(f"    Δ vs paper  : {sym_pap}{delta_pap:+.2f}%")

    # Confusion matrix
    cm  = confusion_matrix(yt, yp, normalize='true',
                           labels=list(range(N_CLS))) * 100
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt=".1f", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, vmin=0, vmax=100)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(
        f"Subject {sid} | Ensemble ({N_ENSEMBLE}×SWCNet)\n"
        f"Acc={te_acc*100:.2f}%  F1={te_f1:.4f}"
    )
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"cm_s{sid}.png"),
                dpi=150, bbox_inches='tight')
    plt.close()

    print("\n" + classification_report(
        yt, yp, target_names=CLASS_NAMES, zero_division=0
    ))

    del Xtr_f, Xev_f, test_models
    return te_acc, te_f1


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    if not os.path.isdir(DATA_DIR):
        raise FileNotFoundError(
            f"\nDataset not found: '{DATA_DIR}'\n"
            f"Update DATA_DIR at the top of this script."
        )

    print("\n" + "="*65)
    print(f"  SWCNet v4  |  Ensemble ({N_ENSEMBLE}×SWCNet)  |  Setup II")
    print("  Same proven v2 architecture, ensemble reduces variance")
    print("="*65)

    results = {}
    for sid in range(1, N_SUBJECTS + 1):
        acc, f1 = setup_II_ensemble(sid, DATA_DIR)
        results[sid] = (acc, f1)

    # ── Final summary table ───────────────────────────────────────────────
    print("\n" + "="*72)
    print(f"  FINAL RESULTS — v4 Ensemble ({N_ENSEMBLE}×SWCNet) vs v2 vs Paper")
    print("="*72)
    print(f"  {'Subj':<6}| {'Paper':>7} | {'v2 Base':>8} | "
          f"{'v4 Ens':>8} | {'Δ Paper':>9} | {'Δ v2':>8}")
    print("-"*72)

    for sid in range(1, N_SUBJECTS + 1):
        v4  = results[sid][0] * 100
        v2  = V2_ACC[sid]
        pap = PAPER_ACC[sid]
        sp  = "✅" if v4 > pap else "⚠️ "
        sb  = "✅" if v4 > v2  else "⚠️ "
        print(f"  S{sid:<5}| {pap:>6.2f}% | {v2:>7.2f}% | "
              f"{v4:>7.2f}% | {sp}{v4-pap:>+7.2f}% | "
              f"{sb}{v4-v2:>+6.2f}%")

    v4_avg  = np.mean([results[s][0]*100 for s in range(1, 10)])
    v2_avg  = np.mean(list(V2_ACC.values()))
    pap_avg = np.mean(list(PAPER_ACC.values()))
    f1_avg  = np.mean([results[s][1]     for s in range(1, 10)])

    print("-"*72)
    print(f"  {'Avg':<6}| {pap_avg:>6.2f}% | {v2_avg:>7.2f}% | "
          f"{v4_avg:>7.2f}% | {v4_avg-pap_avg:>+8.2f}% | "
          f"{v4_avg-v2_avg:>+7.2f}%")
    print(f"  Avg F1 (v4): {f1_avg:.4f}")
    print("="*72)

    # Summary bar chart
    sids     = list(range(1, N_SUBJECTS + 1))
    v4_list  = [results[s][0]*100 for s in sids]
    v2_list  = [V2_ACC[s]         for s in sids]
    pap_list = [PAPER_ACC[s]      for s in sids]

    x = np.arange(N_SUBJECTS); w = 0.25
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(x-w, pap_list, w, label="Paper",       color='gray',       alpha=0.8)
    ax.bar(x,   v2_list,  w, label="v2 Baseline",  color='steelblue',  alpha=0.8)
    ax.bar(x+w, v4_list,  w, label=f"v4 Ensemble", color='darkorange', alpha=0.8)
    for acc, col, lbl in [
        (pap_avg, 'gray',       f"Paper avg: {pap_avg:.1f}%"),
        (v2_avg,  'steelblue',  f"v2 avg:    {v2_avg:.1f}%"),
        (v4_avg,  'darkorange', f"v4 avg:    {v4_avg:.1f}%"),
    ]:
        ax.axhline(acc, color=col, linestyle='--', lw=1.5, label=lbl)
    ax.set_xticks(x)
    ax.set_xticklabels([f"S{s}" for s in sids])
    ax.set_ylabel("Accuracy (%)")
    ax.set_ylim(0, 110)
    ax.set_title(
        f"Setup II: v4 Ensemble ({N_ENSEMBLE}×SWCNet) vs v2 vs Paper "
        f"(BCIC-IV-2a)"
    )
    ax.legend(fontsize=9, ncol=3)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "summary.png"),
                dpi=150, bbox_inches='tight')
    plt.close()
    print(f"\n  Results saved to: {RESULTS_DIR}/")

Device     : cuda
DataDir    : /content/drive/MyDrive/BCICIV_2a_gdf
N_ENSEMBLE : 5 models per fold

  SWCNet v4  |  Ensemble (5×SWCNet)  |  Setup II
  Same proven v2 architecture, ensemble reduces variance

  Subject 1  |  Ensemble SWCNet  (5 models/fold)  Setup II
  [train] X=(288, 22, 1125)  y=(288,)  classes=[0 1 2 3]
  [eval] X=(288, 22, 1125)  y=(288,)  classes=[0 1 2 3]
  Bandpass filtering (4-40 Hz)...

  ── Fold 1/10 ──────────────────────────────────
  Training model 1/5 (seed=420)...
    [S1 f1] m420 Ep  1 | Tr 1.3416/33.3% Va 1.1637/37.9%
    [S1 f1] m420 Ep 20 | Tr 0.0742/98.5% Va 0.7464/65.5%
    [S1 f1] m420 Ep 40 | Tr 0.0338/99.2% Va 0.4449/75.9%
    [S1 f1] m420 Ep 60 | Tr 0.0123/99.8% Va 0.4022/82.8%
    [S1 f1] m420 early stop @ ep77
  Model 1 | Va Acc=82.76%  F1=0.8183
  Training model 2/5 (seed=421)...
    [S1 f1] m421 Ep  1 | Tr 1.2470/40.3% Va 0.9836/48.3%
    [S1 f1] m421 Ep 20 | Tr 0.1117/97.5% Va 0.4431/82.8%
    [S1 f1] m421 Ep 40 | Tr 0.0406/99.2% Va 0.3991/8